Some of the content here is adapted from [LangChain Handbook](https://github.com/pinecone-io/examples/tree/master/generation/langchain/handbook)

# Agents 🤖

Agents are developed on top of LLMs to extend their capabilities. For example, agents provide tools to allow a LLM to access Google search, perform complex calculations with Python, and even make SQL queries.

**Definition**: The key behind agents is giving LLM's the possibility of using tools in their workflow. This is where langchain departs from the popular chatgpt implementation and we can start to get a glimpse of what it offers us as builders. Until now, we covered several building blocks in isolation. Let's see them come to life.

The official definition of agents is the following:


> Agents use an LLM to determine which actions to take and in what order. An action can either be using a tool and observing its output, or returning to the user.

In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

In [2]:
llm = ChatOpenAI(temperature=0, model="gpt-5-mini")

In [3]:
# try to accomplish a task without agents and tools
res = llm.invoke("Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word.")

In [4]:
print(res.content)

Reversed word order: "best the is Hani"

Then reverse characters of each word: "tseb eht si inaH"


In [5]:
@tool
def word_reverse(text: str) -> str:
    """Reverse the order of the words in a sentence."""
    return " ".join(text.split()[::-1])

@tool
def character_reverse(text: str) -> str:
    """Reverse the characters in a string."""
    return text[::-1]

In [6]:
agent1 = create_agent(
    model=llm,
    tools=[word_reverse, character_reverse],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

In [7]:
prompt = "Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word."
result = agent1.invoke({"messages": [{"role": "user", "content": prompt}]})

In [8]:
for m in result['messages']:
    print(m)
    print()

content="Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word." additional_kwargs={} response_metadata={} id='aff1413d-8bdd-4c08-9826-6259b2839d81'

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 185, 'total_tokens': 532, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DFlKqfR79v9JMUQkaKqUjvtGBzTAi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cba20-f64e-7ce3-a615-88ceaf1e7085-0' tool_calls=[{'name': 'word_reverse', 'args': {'text': 'Hani is the best'}, 'id': 'call_c8khQ5q0N15MWAZXjHKs6Cc1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'i

## Example 2: tax assistant agent

In [9]:
def calculate_tax_irs(income):
    # Define the tax brackets and rates
    brackets = [(0, 20550, 0.10), (20551, 83550, 0.12), (83551, 178150, 0.22),
                (178151, 340100, 0.24), (340101, 431900, 0.32),
                (431901, 647850, 0.35), (647851, float('inf'), 0.37)]
    
    tax = 0
    for lower, upper, rate in brackets:
        if income > upper:
            tax += (upper - lower) * rate
        else:
            tax += (income - lower) * rate
            break

    return tax

In [15]:
@tool
def tax_resident(income: float) -> float:
    """Calculate the tax liability of residents. Input: income as a number."""
    print("-----> CALCULATING TAX RESIDENT ON INCOME", income)
    return calculate_tax_irs(income)

@tool
def tax_nonresident(income: float) -> float:
    """Calculate the tax liability of nonresidents. Input: income as a number."""
    print("-----> CALCULATING TAX NON RESIDENT ON INCOME", income)
    return income * 0.25

In [16]:
# https://www.irs.gov/individuals/international-taxpayers/determining-an-individuals-tax-residency-status
AGENT_PROMPT = """You are a tax assistant helping your creator to estimate the tax liability of employees.

First determine the residency status of the client. Below is the IRS guidelines.

'If you are not a U.S. citizen, you are considered a nonresident of the United States for U.S. tax purposes unless you meet one of two tests. You are a resident  of the United States for tax purposes if you meet either the green card test or the substantial presence test for the calendar year (January 1 – December 31).'

Once you determine the residency status, use your tools to calculate the tax liability depending on whether the client is a resident or nonresident.

"""

In [17]:
tax_agent = create_agent(
    model=llm,
    tools=[tax_resident, tax_nonresident],
    system_prompt=AGENT_PROMPT,
)

In [18]:
def ask_tax(client: str):
    result = tax_agent.invoke({"messages": [{"role": "user", 
                                             "content": "Client Information: "+client}]})
    return result["messages"][-1].content


In [19]:
print(ask_tax("John Maden made $100,000 he is US citizen"))

-----> CALCULATING TAX RESIDENT ON INCOME 100000.0
Residency status: U.S. citizen — treated as a U.S. tax resident.

Estimated federal tax liability on $100,000 taxable income: $13,233.66

Notes:
- This estimate uses the resident tax calculation tool. It assumes $100,000 is taxable income after deductions and standard credits; actual tax may differ if itemized deductions, pre-tax retirement contributions, credits, or other adjustments apply.
- State tax and other payroll taxes (Social Security, Medicare) are not included.
- If you want a more accurate estimate, provide filing status (single/married filing jointly, etc.), dependents, pre-tax deductions, or whether $100,000 is gross or taxable income.


In [20]:
print(ask_tax("John Maden is a foreign worker who worked three months as a contractor and made $100,000"))

I need one quick clarification before calculating taxes.

Questions:
1) Is John a U.S. citizen or a green card holder? (If yes, he’s a U.S. tax resident.)
2) How many days was John physically present in the U.S. in the calendar year in question? (If 183+ days by the substantial presence test, he’s a U.S. tax resident.)
3) Was the $100,000 earned for work performed in the U.S. (i.e., U.S.-source income)? 
4) Any tax treaty between the U.S. and John’s country that might affect taxation?

If you don’t want to answer all of those, note that based on the facts you gave (foreign worker, worked three months), he’s most likely a nonresident for U.S. tax purposes (3 months ≈ ~90 days < 183 and assuming no green card). If you want, I can proceed now with a tax estimate assuming he is a nonresident and the $100,000 is U.S.-source. Do you want me to run that estimate?


In [21]:
print(ask_tax("John Maden is a foreign worker (no green card) who worked three months as a contractor and made $100,000"))

-----> CALCULATING TAX NON RESIDENT ON INCOME 100000.0
Summary and residency determination
- You said John Maden is a foreign worker with no green card who worked three months as a contractor and earned $100,000.
- Under IRS rules he is a U.S. tax resident only if he meets the green card test or the substantial presence test (SPT). The SPT requires 31+ days in the current year and a weighted total of 183+ days (current year days + 1/3 of prior year days + 1/6 of second prior year days).
- Working “three months” (~≈90 days) in the U.S. is ordinarily below the 183‑day threshold, so based on the information given he is most likely a nonresident alien for U.S. tax purposes. (If he actually spent 183+ days in the U.S. this calendar year or has other years’ days that push the weighted total to 183+, he could be a resident — see questions below.)

Estimated tax liability (nonresident)
- Using the nonresident calculation, the estimated U.S. tax liability on the $100,000 is: $25,000.
- This est

In [22]:
print(ask_tax("name='John Maden' salary='100000' status='nonresident'"))

-----> CALCULATING TAX NON RESIDENT ON INCOME 100000.0
Residency determination: Client reported status = nonresident. Based on the client's statement, I will treat them as a nonresident for U.S. tax purposes.

Estimated federal tax liability (nonresident):
- Income: $100,000
- Estimated federal tax: $25,000

Notes:
- This is an estimate based on the provided income and assumed nonresident tax rules.
- I did not apply any deductions, treaty benefits, or state taxes. If the client has treaty provisions, exemptions, dependents, itemized deductions, or state tax obligations, the liability may change.
- Recommend reviewing the client's actual residency (green card or substantial presence) and providing additional details (withholding, treaty position, deductions) for a more accurate calculation.


# DataFrame agent
https://python.langchain.com/docs/integrations/toolkits/pandas

In [18]:
import langchain, langchain_experimental
print(langchain.__version__)
print(langchain_experimental.__version__)

1.2.0
0.4.1


In [19]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

In [20]:
df = pd.read_csv("data/mpg.csv")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


In [21]:
agent2 = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling",
)

In [22]:
agent2.invoke("how many rows are there?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape[0]'}`


398There are 398 rows in the dataframe.

> Finished chain.


{'input': 'how many rows are there?',
 'output': 'There are 398 rows in the dataframe.'}

In [24]:
agent2.invoke("how many rows are there?",)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape'}`


(398, 9)There are 398 rows (df.shape[0] == 398). You can also get this with len(df).

> Finished chain.


{'input': 'how many rows are there?',
 'output': 'There are 398 rows (df.shape[0] == 398). You can also get this with len(df).'}

In [ ]:
agent2.run("what country has the highest average mpg?")

In [25]:
agent2.run("what country has the highest average mpg?",)

/var/folders/_c/x7f8prjd14qc082zvt2dfxl80000gn/T/ipykernel_14400/2952901232.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  agent2.run("what country has the highest average mpg?",)




> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('origin')['mpg'].mean().sort_values(ascending=False)"}`


origin
japan     30.450633
europe    27.891429
usa       20.083534
Name: mpg, dtype: float64Japan has the highest average mpg (about 30.45 mpg).

> Finished chain.


'Japan has the highest average mpg (about 30.45 mpg).'

# SQL Database

https://python.langchain.com/docs/integrations/toolkits/sql_database

In [26]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

In [27]:
db = SQLDatabase.from_uri("sqlite:///data/ClassicModels.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

/Users/hanisaf/.local/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:159: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['11']; using no arguments.
  self._metadata.reflect(
/Users/hanisaf/.local/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:159: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.SMALLINT'> with reflected arguments ['6']; using no arguments.
  self._metadata.reflect(


In [28]:
sql_tools = toolkit.get_tools()
sql_tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x318f6d7f0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x318f6d7f0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x318f6d7f0>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [29]:
sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=(
        "You are an agent designed to interact with a SQL database. "
        "Use the provided tools to inspect schema and run queries safely."
    ),
)

In [30]:
resp = sql_agent.invoke({"messages": [{"role": "user", "content": "List the tables in the database."}]})
print(resp["messages"][-1].content)

The database contains these tables:
- Customers
- Employees
- Offices
- OrderDetails
- Orders
- Payments
- ProductLines
- Products

Would you like the schema or sample rows for any of these tables?


In [31]:
resp

{'messages': [HumanMessage(content='List the tables in the database.', additional_kwargs={}, response_metadata={}, id='58065157-73e6-4639-8209-a9c5cf0b39d4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 433, 'total_tokens': 454, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CqnSLnVvOC5QsBYwDXwPbwy7xuHCy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b577a-7363-7213-9b48-47ed4428c143-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {}, 'id': 'call_ZM47naB0CC1F6HkPdNUT7Zhx', 'type': 'tool_call'}], usage_metadata={'input_tokens': 433, 'output_tokens': 21, 'total_tokens': 454, 'i

In [32]:
resp = sql_agent.invoke({"messages": [{"role": "user", "content": "Select the first five rows in Employees."}]})
print(resp["messages"][-1].content)

Here are the first five rows from Employees (ordered by employeeNumber):

1) employeeNumber: 1002, lastName: Murphy, firstName: Diane, extension: x5800, email: dmurphy@classicmodelcars.com, reportsTo: 0, jobTitle: President, officeCode: 1
2) employeeNumber: 1056, lastName: Patterson, firstName: Mary, extension: x4611, email: mpatterso@classicmodelcars.com, reportsTo: 1002, jobTitle: VP Sales, officeCode: 1
3) employeeNumber: 1076, lastName: Firrelli, firstName: Jeff, extension: x9273, email: jfirrelli@classicmodelcars.com, reportsTo: 1002, jobTitle: VP Marketing, officeCode: 1
4) employeeNumber: 1088, lastName: Patterson, firstName: William, extension: x4871, email: wpatterson@classicmodelcars.com, reportsTo: 1056, jobTitle: Sales Manager (APAC), officeCode: 6
5) employeeNumber: 1102, lastName: Bondur, firstName: Gerard, extension: x5408, email: gbondur@classicmodelcars.com, reportsTo: 1056, jobTitle: Sale Manager (EMEA), officeCode: 4


In [33]:
resp

{'messages': [HumanMessage(content='Select the first five rows in Employees.', additional_kwargs={}, response_metadata={}, id='5c77cc31-8d4f-4966-bf88-963d3e4f05e9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 434, 'total_tokens': 583, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CqnSPoBVs2mxEyn8W9Jkt6m3gsd9S', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b577a-83b6-7ac2-bc7f-8d391bbd6c89-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {}, 'id': 'call_PsA8ZqG2jUjbpmVeiiLz4WlS', 'type': 'tool_call'}], usage_metadata={'input_tokens': 434, 'output_tokens': 149, 'total_toke